In [ ]:
# copy NHL SCHEDULE FOR TODAY AND MAKE NEW FILE FOR INDEX.HTML TO READ

import importlib
import script
import NHL_script
import NHL_script


# NHL SCHEDULE
NHL_script.make_todays_schedule()



Today's schedule copied to NHL_data/NHL_todays_schedule.txt


AttributeError: 'Response' object has no attribute 'get'

In [ ]:
from datetime import datetime, timedelta
import json
import csv
import requests
from pathlib import Path  # Import Path

def generate_hockey_reference_link(name):
    """
    Generates a Hockey Reference player link based on the player's name.

    Args:
        name (str): The player's full name in the format "FirstName LastName".

    Returns:
        str: The Hockey Reference player link.
    """
    # Split the name into first and last names
    try:
        first_name, last_name = name.split(" ")
    except ValueError:
        return "Invalid name format. Expected 'FirstName LastName'."

    # Extract the first letter of the last name
    last_name_initial = last_name[0].lower()

    # Extract the first two letters of the first name
    first_name_initials = first_name[:2].lower()

    # Format the link
    link = f"https://www.hockey-reference.com/players/{last_name_initial}/{last_name[:5].lower()}{first_name_initials}01.html"

    return link

def process_yesterdays_scores_to_report():
    """
    Fetches yesterday's NHL scores from the NHL API, processes the data,
    """
    # Get yesterday's date in the required format
    yesterdays_date = (datetime.now() - timedelta(days=1)).strftime("%Y-%m-%d")
    url = f"https://api-web.nhle.com/v1/score/{yesterdays_date}"
    print(url)
    resp = requests.get(
            url,
            timeout=30,
            allow_redirects=True,
            headers={"Accept": "application/json"},
        )
    resp.raise_for_status()
    resp.json()

    # File paths
    output_dir = Path("NHL_data/daily_scores")  # Convert to Path object

    data = resp.json()

    # Prepare the output file path
    output_file = f"NHL_data/daily_scores/NHL_scores_{yesterdays_date}.json"

    # Ensure the output directory exists
    output_dir.mkdir(parents=True, exist_ok=True)

    # Extract and format the games data
    formatted_games = []
    for game in data.get("games", []):
        home_team = game["homeTeam"]["name"]["default"]
        away_team = game["awayTeam"]["name"]["default"]
        home_score = game["homeTeam"]["score"]
        away_score = game["awayTeam"]["score"]
        winner = home_team if home_score > away_score else away_team
        condensed_game = game.get("condensedGame", "")
        condensed_game = "https://www.nhl.com" + condensed_game

        # Extract goals data
        goals = []
        for goal in game.get("goals", []):
            goal_data = {
                "player_id": goal["playerId"],
                "name": f"{goal['firstName']['default']} {goal['lastName']['default']}",
                "team": goal["teamAbbrev"],
                "goals_to_date": goal.get("goalsToDate", None),
                "assists": [
                    {
                        "name": assist["name"]["default"],
                        "assists_to_date": assist["assistsToDate"],
                        "player_id": assist["playerId"],
                    }
                    for assist in goal.get("assists", [])
                ],
            }
            goals.append(goal_data)

        # Add the formatted game data
        formatted_games.append(
            {
                "date": yesterdays_date,
                "home_team": home_team,
                "away_team": away_team,
                "home_score": home_score,
                "away_score": away_score,
                "winner": winner,
                "condensed_game": condensed_game,
                "goals": goals,
            }
        )

    # Save the formatted data to the output file
    with open(output_file, "w", encoding="utf-8") as file:
        json.dump(formatted_games, file, indent=4)

    print(f"Processed scores saved to {output_file}")

    # Run the function
    yesterdays_scores = formatted_games

    # Generate the report
    data = yesterdays_scores
    """
    Processes a JSON string of NHL scores and generates an HTML report.

    Args:
        data (list): List of games containing NHL scores data.

    Returns:
        str: A formatted HTML report of the games and their details.
    """
    # Initialize the report
    report_lines = []

    # Extract the date from the first game (assuming all games are from the same date)
    if data:
        report_lines.append(f"<h2>DATE: {data[0]['date']}</h2>")
    else:
        return "<p>No games available to report.</p>"

    # Process each game
    for i, game in enumerate(data, start=1):
        # Add match header with video link
        report_lines.append(
            f"<h2>MATCH {i}: <a target='_blank' rel='noopener noreferrer' href='{game['condensed_game']}'>Video</a></h2>"
        )
        report_lines.append(
            f"<h2>{game['home_team']} {game['home_score']} vs {game['away_team']} {game['away_score']}</h2>"
        )

        # Start the table
        report_lines.append(
            "<table border='1' style='border-collapse: collapse; width: 100%;'>"
        )
        report_lines.append(
            "<tr><th>Team</th><th>Name</th><th>Assist1</th><th>Assist2</th></tr>"
        )

        # Process each goal
        for goal in game.get("goals", []):
            # Extract assists
            assists = goal.get("assists", [])
            assist1 = (
                f"{assists[0]['name']} ({assists[0].get('assists_to_date', 'N/A')})"
                if len(assists) > 0
                else ""
            )
            assist2 = (
                f"{assists[1]['name']} ({assists[1].get('assists_to_date', 'N/A')})"
                if len(assists) > 1
                else ""
            )

            # Add a row for the goal
            report_lines.append(
                f"<tr>"
                f"<td>{goal['team']}</td>"
                f"<td><a target='_blank' rel='noopener noreferrer' href='{generate_hockey_reference_link(goal['name'])}'>{goal['name']}</a> ({goal.get('goals_to_date', 'N/A')})</td>"
                f"<td>{assist1}</td>"
                f"<td>{assist2}</td>"
                f"</tr>"
            )

        # End the table
        report_lines.append("</table>")
        report_lines.append("<br>")  # Add spacing between matches

    # Join the report lines into a single HTML string
    report = "\n".join(report_lines)
    # Print the report
    # print(report)

    # Optionally, save the report to a text file
    with open("NHL_data/NHL_yesterdays_scores.txt", "w") as file:
        file.write(report)
    print("report generated")

process_yesterdays_scores_to_report()


https://api-web.nhle.com/v1/score/2025-10-16
Processed scores saved to NHL_data/daily_scores/NHL_scores_2025-10-16.json


In [ ]:
import importlib
import NHL_script
importlib.reload(NHL_script)
teams_list = NHL_script.teams_today()
print(teams_list)

['ANA', 'CAR', 'CBJ', 'COL', 'DAL', 'VAN', 'LAK', 'PIT', 'MTL', 'NSH', 'NJD', 'FLA', 'NYI', 'EDM', 'OTT', 'SEA', 'PHI', 'WPG', 'TOR', 'NYR', 'VGK', 'BOS']


In [ ]:
# getting all players from the rosters to compare against skaters

import importlib
import NHL_script
importlib.reload(NHL_script)
import NHL_script
importlib.reload(NHL_script)
import file_operations
importlib.reload(file_operations)

from collections import defaultdict

teams_list = NHL_script.teams_today()
# NHL_script.update_rosters()

# ids for players on the teams playing today
roster_ids = []
for x in teams_list:
    roster = NHL_script.get_roster(x)
    for x in roster[1:]:
        roster_ids.append(x[9])

# get a list of skater paths

# Example usage
directory = "NHL_data/daily_skaters"
file_paths = NHL_data.get_sorted_skater_paths(directory)
list11 = []
for x in file_paths:
    csv_file_path = x
    file1 = file_operations.read_csv(csv_file_path)
    list11.append(file1)
    
list_of_data = []
for z in roster_ids:
    for x in list11:
        for y in x:
            if z == y[0] and y[5] == 'all':
                list_of_data.append(y)

# Combine rows by ID
combined_data = defaultdict(lambda: defaultdict(set))  # Use sets to ensure unique values

for row in list_of_data:
    player_id = row[0]
    for i, value in enumerate(row):
        if i > 6:  # Convert numeric values to floats and store in sets
            combined_data[player_id][i].add(float(value))
        else:  # Store non-numeric values in sets
            combined_data[player_id][i].add(value)

# Convert sets to sorted lists for the final output
final_data = []
for player_id, columns in combined_data.items():
    combined_row = []
    for i in range(len(list_of_data[0])):
        if i in columns:
            if i > 6:  # Sort numeric values in descending order
                combined_row.append(sorted(columns[i], reverse=True))
            else:  # Sort non-numeric values (if needed)
                combined_row.append(sorted(columns[i], reverse=True))
        else:
            combined_row.append([])
    final_data.append(combined_row)


import csv
from datetime import datetime

# Define the headers
headers = [
    "playerId", "season", "name", "team", "position", "situation", "games_played", "icetime", "shifts", "gameScore",
    "onIce_xGoalsPercentage", "offIce_xGoalsPercentage", "onIce_corsiPercentage", "offIce_corsiPercentage",
    "onIce_fenwickPercentage", "offIce_fenwickPercentage", "iceTimeRank", "I_F_xOnGoal", "I_F_xGoals", "I_F_xRebounds",
    "I_F_xFreeze", "I_F_xPlayStopped", "I_F_xPlayContinuedInZone", "I_F_xPlayContinuedOutsideZone",
    "I_F_flurryAdjustedxGoals", "I_F_scoreVenueAdjustedxGoals", "I_F_flurryScoreVenueAdjustedxGoals",
    "I_F_primaryAssists", "I_F_secondaryAssists", "I_F_shotsOnGoal", "I_F_missedShots", "I_F_blockedShotAttempts",
    "I_F_shotAttempts", "I_F_points", "I_F_goals", "I_F_rebounds", "I_F_reboundGoals", "I_F_freeze", "I_F_playStopped",
    "I_F_playContinuedInZone", "I_F_playContinuedOutsideZone", "I_F_savedShotsOnGoal", "I_F_savedUnblockedShotAttempts",
    "penalties", "I_F_penalityMinutes", "I_F_faceOffsWon", "I_F_hits", "I_F_takeaways", "I_F_giveaways",
    "I_F_lowDangerShots", "I_F_mediumDangerShots", "I_F_highDangerShots", "I_F_lowDangerxGoals",
    "I_F_mediumDangerxGoals", "I_F_highDangerxGoals", "I_F_lowDangerGoals", "I_F_mediumDangerGoals",
    "I_F_highDangerGoals", "I_F_scoreAdjustedShotsAttempts", "I_F_unblockedShotAttempts",
    "I_F_scoreAdjustedUnblockedShotAttempts", "I_F_dZoneGiveaways", "I_F_xGoalsFromxReboundsOfShots",
    "I_F_xGoalsFromActualReboundsOfShots", "I_F_reboundxGoals", "I_F_xGoals_with_earned_rebounds",
    "I_F_xGoals_with_earned_rebounds_scoreAdjusted", "I_F_xGoals_with_earned_rebounds_scoreFlurryAdjusted",
    "I_F_shifts", "I_F_oZoneShiftStarts", "I_F_dZoneShiftStarts", "I_F_neutralZoneShiftStarts", "I_F_flyShiftStarts",
    "I_F_oZoneShiftEnds", "I_F_dZoneShiftEnds", "I_F_neutralZoneShiftEnds", "I_F_flyShiftEnds", "faceoffsWon",
    "faceoffsLost", "timeOnBench", "penalityMinutes", "penalityMinutesDrawn", "penaltiesDrawn", "shotsBlockedByPlayer",
    "OnIce_F_xOnGoal", "OnIce_F_xGoals", "OnIce_F_flurryAdjustedxGoals", "OnIce_F_scoreVenueAdjustedxGoals",
    "OnIce_F_flurryScoreVenueAdjustedxGoals", "OnIce_F_shotsOnGoal", "OnIce_F_missedShots",
    "OnIce_F_blockedShotAttempts", "OnIce_F_shotAttempts", "OnIce_F_goals", "OnIce_F_rebounds",
    "OnIce_F_reboundGoals", "OnIce_F_lowDangerShots", "OnIce_F_mediumDangerShots", "OnIce_F_highDangerShots",
    "OnIce_F_lowDangerxGoals", "OnIce_F_mediumDangerxGoals", "OnIce_F_highDangerxGoals", "OnIce_F_lowDangerGoals",
    "OnIce_F_mediumDangerGoals", "OnIce_F_highDangerGoals", "OnIce_F_scoreAdjustedShotsAttempts",
    "OnIce_F_unblockedShotAttempts", "OnIce_F_scoreAdjustedUnblockedShotAttempts", "OnIce_F_xGoalsFromxReboundsOfShots",
    "OnIce_F_xGoalsFromActualReboundsOfShots", "OnIce_F_reboundxGoals", "OnIce_F_xGoals_with_earned_rebounds",
    "OnIce_F_xGoals_with_earned_rebounds_scoreAdjusted", "OnIce_F_xGoals_with_earned_rebounds_scoreFlurryAdjusted",
    "OnIce_A_xOnGoal", "OnIce_A_xGoals", "OnIce_A_flurryAdjustedxGoals", "OnIce_A_scoreVenueAdjustedxGoals",
    "OnIce_A_flurryScoreVenueAdjustedxGoals", "OnIce_A_shotsOnGoal", "OnIce_A_missedShots",
    "OnIce_A_blockedShotAttempts", "OnIce_A_shotAttempts", "OnIce_A_goals", "OnIce_A_rebounds",
    "OnIce_A_reboundGoals", "OnIce_A_lowDangerShots", "OnIce_A_mediumDangerShots", "OnIce_A_highDangerShots",
    "OnIce_A_lowDangerxGoals", "OnIce_A_mediumDangerxGoals", "OnIce_A_highDangerxGoals", "OnIce_A_lowDangerGoals",
    "OnIce_A_mediumDangerGoals", "OnIce_A_highDangerGoals", "OnIce_A_scoreAdjustedShotsAttempts",
    "OnIce_A_unblockedShotAttempts", "OnIce_A_scoreAdjustedUnblockedShotAttempts", "OnIce_A_xGoalsFromxReboundsOfShots",
    "OnIce_A_xGoalsFromActualReboundsOfShots", "OnIce_A_reboundxGoals", "OnIce_A_xGoals_with_earned_rebounds",
    "OnIce_A_xGoals_with_earned_rebounds_scoreAdjusted", "OnIce_A_xGoals_with_earned_rebounds_scoreFlurryAdjusted",
    "OffIce_F_xGoals", "OffIce_A_xGoals", "OffIce_F_shotAttempts", "OffIce_A_shotAttempts", "xGoalsForAfterShifts",
    "xGoalsAgainstAfterShifts", "corsiForAfterShifts", "corsiAgainstAfterShifts", "fenwickForAfterShifts",
    "fenwickAgainstAfterShifts"
]

# Get today's date
todays_date = datetime.now().strftime('%Y-%m-%d')

# Define the output file path
output_file = f'NHL_data/combined_skaters/combined_skaters_{todays_date}.csv'

# Write the data to the CSV file
with open(output_file, mode='w', newline='', encoding='utf-8') as file:
    writer = csv.writer(file)
    
    # Write the headers
    writer.writerow(headers)
    
    # Write the rows
    writer.writerows(final_data)

print(f"Data has been saved to {output_file}")


Data has been saved to NHL_data/combined_skaters/combined_skaters_2025-10-16.csv


In [ ]:
import csv
import json

# Define the input CSV file and output JSON file paths
csv_file_path = 'NHL_data/combined_skaters/combined_skaters_2025-10-15.csv'
json_file_path = 'NHL_data/combined_skaters/combined_skaters_2025-10-15.json'

# Read the CSV file and convert it to a list of dictionaries
data = []
with open(csv_file_path, mode='r', encoding='utf-8') as csv_file:
    csv_reader = csv.DictReader(csv_file)  # Automatically uses the headers as keys
    for row in csv_reader:
        # Convert any string representations of lists back to Python lists
        for key, value in row.items():
            if value.startswith('[') and value.endswith(']'):
                try:
                    row[key] = json.loads(value)  # Parse the string as a list
                except json.JSONDecodeError:
                    pass  # Leave the value as is if it can't be parsed
        data.append(row)

# Write the data to a JSON file
with open(json_file_path, mode='w', encoding='utf-8') as json_file:
    json.dump(data, json_file, indent=4)

print(f"CSV data has been converted to JSON and saved to {json_file_path}")

CSV data has been converted to JSON and saved to NHL_data/combined_skaters_2025-10-15.json


In [6]:
import pandas as pd

def csv_to_html(csv_file_path):
    """
    Reads a CSV file and converts it to an HTML file with the same name but with a .html extension.

    Args:
        csv_file_path (str): Path to the CSV file.

    Returns:
        str: Path to the generated HTML file.
    """
    # Generate the HTML file path
    html_file_path = csv_file_path.replace('.csv', '.html')

    # Read the CSV file into a DataFrame
    df = pd.read_csv(csv_file_path)

    # Convert the DataFrame to an HTML file
    df.to_html(html_file_path, index=False)

    print(f"HTML file has been created: {html_file_path}")
    return html_file_path

# Example usage
csv_file_path = 'NHL_data/combined_skaters_2025-10-15.csv'
html_file_path = csv_to_html(csv_file_path)

HTML file has been created: NHL_data/combined_skaters_2025-10-15.html


In [ ]:
import json

def read_json_to_list(json_file_path):
    """
    Reads a JSON file and returns its contents as a list of dictionaries.

    Args:
        json_file_path (str): Path to the JSON file.

    Returns:
        list: List of dictionaries containing the JSON data.
    """
    with open(json_file_path, mode='r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    return data

# Example usage
json_file_path = 'NHL_data/combined_skaters_2025-10-15.json'
data = read_json_to_list(json_file_path)

# # Print the first entry to verify
# print(data[0] if data else "No data found")


# for x in data:
#     if str(x.get('playerId')) == "['8484153']":  # Convert to string for comparison
#         print(x)

# for x in data:
#     id1 = x.get('playerId')
#     print(id1)
for x in data:
    print('SOG:', x.get('I_F_shotsOnGoal'),x.get('name'), x.get('team'), 'G:', x.get('I_F_goals'),'P:', x.get('I_F_points'))

SOG: [2.0] ['Justin Danforth'] ['BUF'] G: [0.0] P: [0.0]
SOG: [9.0, 8.0, 5.0, 2.0] ['Josh Doan'] ['BUF'] G: [0.0] P: [0.0]
SOG: [0.0] ['Mason Geertsen'] ['BUF'] G: [0.0] P: [0.0]
SOG: [1.0, 0.0] ['Tyson Kozak'] ['BUF'] G: [0.0] P: [0.0]
SOG: [3.0, 0.0] ['Peyton Krebs'] ['BUF'] G: [0.0] P: [0.0]
SOG: [7.0, 3.0, 2.0] ['Jiri Kulich'] ['BUF'] G: [0.0] P: [0.0]
SOG: [3.0, 1.0] ['Beck Malenstyn'] ['BUF'] G: [0.0] P: [0.0]
SOG: [2.0] ['Ryan McLeod'] ['BUF'] G: [0.0] P: [0.0]
SOG: [4.0] ['Josh Norris'] ['BUF'] G: [0.0] P: [0.0]
SOG: [2.0, 1.0] ['Jack Quinn'] ['BUF'] G: [0.0] P: [0.0]
SOG: [18.0, 17.0, 13.0, 5.0] ['Tage Thompson'] ['BUF'] G: [1.0, 0.0] P: [1.0, 0.0]
SOG: [10.0, 9.0, 7.0, 6.0] ['Alex Tuch'] ['BUF'] G: [0.0] P: [1.0, 0.0]
SOG: [9.0, 8.0, 5.0] ['Jason Zucker'] ['BUF'] G: [1.0, 0.0] P: [1.0, 0.0]
SOG: [1.0, 0.0] ['Jacob Bryson'] ['BUF'] G: [0.0] P: [0.0]
SOG: [2.0, 1.0] ['Bowen Byram'] ['BUF'] G: [0.0] P: [0.0]
SOG: [8.0, 4.0] ['Rasmus Dahlin'] ['BUF'] G: [0.0] P: [1.0, 0.0]
SOG: [

In [8]:
import json

def read_json_to_list(json_file_path):
    """
    Reads a JSON file and returns its contents as a list of dictionaries.

    Args:
        json_file_path (str): Path to the JSON file.

    Returns:
        list: List of dictionaries containing the JSON data.
    """
    with open(json_file_path, mode='r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    return data

# Example usage
json_file_path = 'NHL_data/combined_skaters_2025-10-15.json'
data = read_json_to_list(json_file_path)

for key, value in data[0].items():
    print(f"{key}: {type(value)}{value}")

playerId: <class 'str'>['8479941']
season: <class 'str'>['2025']
name: <class 'str'>['Justin Danforth']
team: <class 'str'>['BUF']
position: <class 'str'>['R']
situation: <class 'str'>['all']
games_played: <class 'str'>['3', '2', '1']
icetime: <class 'list'>[2435.0, 1380.0, 573.0]
shifts: <class 'list'>[61.0, 39.0, 18.0]
gameScore: <class 'list'>[0.18, 0.1, -0.56]
onIce_xGoalsPercentage: <class 'list'>[0.39, 0.32, 0.29]
offIce_xGoalsPercentage: <class 'list'>[0.62, 0.45, 0.39]
onIce_corsiPercentage: <class 'list'>[0.53, 0.45, 0.33]
offIce_corsiPercentage: <class 'list'>[0.55, 0.54, 0.53, 0.5]
onIce_fenwickPercentage: <class 'list'>[0.5, 0.4, 0.27]
offIce_fenwickPercentage: <class 'list'>[0.53, 0.5]
iceTimeRank: <class 'list'>[22.0, 19.0, 10.0]
I_F_xOnGoal: <class 'list'>[2.49, 1.65]
I_F_xGoals: <class 'list'>[0.14, 0.06]
I_F_xRebounds: <class 'list'>[0.18, 0.07]
I_F_xFreeze: <class 'list'>[0.44, 0.27]
I_F_xPlayStopped: <class 'list'>[0.1, 0.05]
I_F_xPlayContinuedInZone: <class 'list'>[

In [1]:
import json
import ast
import numpy as np

def analyze_sequence(sequence):
    # Calculate the differential
    differential = np.diff(sequence)
    
    # Normalize the sequence
    min_val, max_val = min(sequence), max(sequence)
    if max_val == min_val:
        # If all values are the same, set normalized values to 0.0
        normalized = [0.0 for _ in sequence]
    else:
        normalized = [(x - min_val) / (max_val - min_val) for x in sequence]
    
    # Calculate variance of the differential
    variance = np.var(differential)
    
    return {
        "original": sequence,
        "differential": differential.tolist(),
        "normalized": normalized,
        "variance_of_differential": variance
    }

# def analyze_sequence(sequence):
#     # Calculate the differential
#     differential = np.diff(sequence)
    
#     # Normalize the sequence
#     min_val, max_val = min(sequence), max(sequence)
#     normalized = [(x - min_val) / (max_val - min_val) for x in sequence]
    
#     # Calculate variance of the differential
#     variance = np.var(differential)
    
#     return {
#         "original": sequence,
#         "differential": differential.tolist(),
#         "normalized": normalized,
#         "variance_of_differential": variance
#     }

def read_json_to_list(json_file_path):
    """
    Reads a JSON file and returns its contents as a list of dictionaries.

    Args:
        json_file_path (str): Path to the JSON file.

    Returns:
        list: List of dictionaries containing the JSON data.
    """
    with open(json_file_path, mode='r', encoding='utf-8') as json_file:
        data = json.load(json_file)
    return data

# Example usage
json_file_path = 'NHL_data/combined_skaters_2025-10-15.json'
data = read_json_to_list(json_file_path)

SOG_csv = []
for x in data:
    # name
    name = x.get('name')
    team = x.get('team')
    # games_played
    games_played_list = x.get('games_played')
    list_of_numbers = [int(x) for x in ast.literal_eval(games_played_list)]
    games_played = list_of_numbers[0]
    # SOG
    SOG_list = x.get('I_F_shotsOnGoal')
    SOG = SOG_list[0]
    #SOG PER GAME
    SOG_per_game = SOG / games_played if games_played > 0 else 0
    
    # analyze SOG_list to find differentials and stuff
    # example output
    # original: [9.0, 7.0, 5.0]
    # differential: [-2.0, -2.0]
    # normalized: [1.0, 0.5, 0.0]
    # variance_of_differential: 0.0

    # print(SOG_list.reverse())
    # result = analyze_sequence(SOG_list.reverse())
    
    # Reverse the list without modifying it in place
    reversed_SOG_list = SOG_list[::-1]
    # print(reversed_SOG_list)
    result = analyze_sequence(reversed_SOG_list)

    SOG_differential = result.get('differential', [99])
    # SOG_normalized = result.get('normalized', [99])
    SOG_variance = result.get('variance_of_differential', 99)

    SOG_csv_entry = []

    SOG_csv_entry.append(name.strip('[]').strip("'"))
    SOG_csv_entry.append(team.strip('[]').strip("'"))
    SOG_csv_entry.append(games_played_list)
    SOG_csv_entry.append(SOG)
    SOG_csv_entry.append(round(SOG_per_game,2))
    SOG_csv_entry.append(SOG_list)
    SOG_csv_entry.append(SOG_differential)
    # SOG_csv_entry.append(SOG_normalized)
    SOG_csv_entry.append(round(SOG_variance,2))

    SOG_csv.append(SOG_csv_entry)




# convert list of lists csv to pandas df
import pandas as pd
df = pd.DataFrame(SOG_csv, columns=['name', 'team', 'GP','SOG', 'SOG_per_game', 'SOG_list','SOG_differential','SOG_variance'])

# save df to csv
csv_file_path = 'NHL_data/SOG_per_game.csv'
df.to_csv(csv_file_path, index=False)
print(f"CSV file has been created: {csv_file_path}")

# convert df to html
html_file_path = 'NHL_data/SOG_per_game.html'
df.to_html(html_file_path, index=False)
print(f"HTML file has been created: {html_file_path}")

/home/codespace/.local/lib/python3.12/site-packages/numpy/_core/fromnumeric.py:4268: RuntimeWarning: Degrees of freedom <= 0 for slice
  return _methods._var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/codespace/.local/lib/python3.12/site-packages/numpy/_core/_methods.py:181: RuntimeWarning: invalid value encountered in divide
  arrmean = um.true_divide(arrmean, div, out=arrmean,
/home/codespace/.local/lib/python3.12/site-packages/numpy/_core/_methods.py:215: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


CSV file has been created: NHL_data/SOG_per_game.csv
HTML file has been created: NHL_data/SOG_per_game.html
